# Statistical Analysis

In this section, we evaluate whether the treatment group leads to a higher
conversion rate compared to the control group.

We use a one-sided two-proportion z-test to determine whether the observed
difference is statistically significant.

In [ ]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath(".."))

df = pd.read_csv("../data/processed/ab_data_cleaned.csv")

df.head()

---
## Conversion Summary

We begin by summarizing the number of conversions and total observations
for each experimental group.

In [ ]:
summary = (
    df.groupby("group")["converted"]
    .agg(["sum", "count"])
)

summary["conversion_rate"] = (
    summary["sum"] / summary["count"]
)

summary

The conversion rate is defined as the proportion of users who converted
in each group.

In [ ]:
conversion_rates = summary["conversion_rate"]

print(f"Control conversion rate: {conversion_rates['control']:.2%}")
print(f"Treatment conversion rate: {conversion_rates['treatment']:.2%}")

The treatment group shows a visibly higher conversion rate than the
control group. We now evaluate whether this observed difference is
statistically significant.

---
## Hypothesis Testing

We perform a one-sided two-proportion z-test to evaluate whether the
treatment group achieves a higher conversion rate than the control group.

$H_0$: $p_{treatment} = p_{control}$

$H_1$: $p_{treatment} > p_{control}$

The treatment group is placed first in the test so that the alternative
hypothesis corresponds to an increase in conversion rate.

The z-test uses the pooled proportion estimator under the null hypothesis.

In [ ]:
from src import conversion_ztest

z_result = conversion_ztest(summary)

pd.DataFrame([z_result])

The p-value is effectively zero (below machine precision), providing
considerably strong evidence against the null hypothesis.

Therefore, we reject $H_0$ and conclude that the treatment group achieves
a higher conversion rate than the control group.

Given the large sample size, even moderate differences are expected to
produce extremely small p-values.

To better understand the numerical scale of the result, we approximate
the order of magnitude of the p-value using the survival function of
the standard normal distribution.

In [ ]:
from scipy.stats import norm

p_value_order = round(norm.logsf(z_result["z_stat"]), 2)

print(f"The p-value's order of magnitude is 10^{p_value_order}")

---
## Effect Size Estimation

Statistical significance alone does not indicate whether the effect is
practically meaningful. We therefore estimate the magnitude of the effect.

In [ ]:
uplift = (
    conversion_rates["treatment"]
    - conversion_rates["control"]
)

relative_uplift = (
    uplift / conversion_rates["control"]
)

print(f"Absolute uplift: {uplift:.4f}")
print(f"Relative uplift: {relative_uplift:.2%}")

The absolute uplift measures the direct increase in conversion rate,
while the relative uplift measures the proportional improvement relative
to the control group.

---
## Confidence Interval

We compute a 95% confidence interval for the difference in conversion
rates between treatment and control groups.

In [ ]:
from src import confidence_interval

ci_result = confidence_interval(conversion_rates, summary)

pd.DataFrame([ci_result])

The confidence interval provides a range of plausible values for the
true treatment effect.

Since the interval does not include zero, the result is consistent with
the conclusion obtained from the hypothesis test.

Additionally, the interval suggests that the true treatment effect remains
positive across all plausible parameter values.

---
## Visualization

In [ ]:
from src import plot_conversion_rates

plot_conversion_rates(conversion_rates)

The plot illustrates the conversion rates observed in each group and
highlights the higher performance of the treatment group.

---
## Guardrail Metric Analysis

While the treatment increases conversion rate, it is also important to
evaluate whether the experiment negatively impacts other business metrics.

We therefore analyze purchase amount as a guardrail metric.

### Revenue per User

To avoid selection bias, we evaluate average purchase amount across all
users, including non-converted users.

This metric can be interpreted as average revenue per user (ARPU).

In [ ]:
revenue_summary = (
    df.groupby("group")["purchase_amount"]
    .agg(["mean", "std", "count"])
)

revenue_summary

We compare the mean purchase amount between groups using Welch's t-test.
This test is appropriate because it does not assume equal variances
between groups.

$H_0$: $\mu_{treatment} = \mu_{control}$

$H_1$: $\mu_{treatment} \neq \mu_{control}$

The p-value indicates whether the observed difference in average revenue
per user is statistically significant.

In [ ]:
from src import t_test

revenue_result = t_test(df, "purchase_amount")

pd.DataFrame([revenue_result])

Again, the p-value is effectively zero, providing
considerably strong evidence against the null hypothesis.

Therefore, we reject $H_0$ and conclude that the average purchase amount
differs between groups.

Since the treatment group presents a higher observed mean purchase amount,
the observed effect is consistent with a positive impact on revenue per user.

In [ ]:
control_revenue = revenue_summary.loc["control", "mean"]
treatment_revenue = revenue_summary.loc["treatment", "mean"]

revenue_diff = treatment_revenue - control_revenue
relative_revenue_diff = revenue_diff / control_revenue

print(f"Control ARPU: {control_revenue:.2f}")
print(f"Treatment ARPU: {treatment_revenue:.2f}")

print(f"\nAbsolute difference: {revenue_diff:.2f}")
print(f"Relative difference: {relative_revenue_diff:.2%}")

The guardrail analysis should be interpreted together with the conversion
results to assess the overall impact of the treatment.

---
## Conclusion

The statistical analysis provides strong evidence that the treatment
group achieves a higher conversion rate than the control group.

The conversion rate increased from 11.87% in the control group to
17.95% in the treatment group, representing an absolute uplift of
approximately 6 percentage points and a relative increase of
approximately 51%.

Additionally, the 95% confidence interval excludes zero, reinforcing
the conclusion that the treatment effect is consistently positive.

The guardrail analysis also indicates a higher average revenue per user
in the treatment group, with ARPU increasing from 4.45 to 6.76.

Taken together, these results suggest that the treatment not only improves
conversion performance, but also produces a positive business impact
without evidence of adverse effects on the evaluated guardrail metric.